# Multi-Modal Precision Agriculture with OmniEarth

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshsk/omniearth/blob/main/examples/01_precision_agriculture_multimodal.ipynb)

Fetch **Sentinel-2 Optical**, **Copernicus DEM**, **ISRIC 3D Soil**, and **Open-Meteo Weather (ET0 & VPD)** for any agricultural AOI in Python with **zero API keys**.

In [ ]:
!pip install -q "omniearth[gis]"

In [ ]:
from omniearth import OmniEarth, compute_spectral_index

client = OmniEarth()
field_bbox = [78.50, 20.65, 78.60, 20.72]  # [min_lon, min_lat, max_lon, max_lat]

# 1. Cloud-filtered Sentinel-2 Level-2A Surface Reflectance
optical = client.search(category="optical", bbox=field_bbox, max_cloud_cover=15, limit=5)
print(f"Found {len(optical)} Sentinel-2 scenes; latest: {optical.first().id}")

# Convert results directly to a GeoPandas GeoDataFrame
gdf = optical.to_geodataframe()
gdf[["id", "datetime", "cloud_cover", "geometry"]].head()

In [ ]:
# 2. Weather (ET0 / VPD), 3D Soil Profile & 30m Elevation (all Zero-Config)
weather = client.get_weather(bbox=field_bbox)
soil = client.get_soil(bbox=field_bbox, depth="0-5cm")
dem = client.get_elevation(bbox=field_bbox, dataset="copernicus")

print("Weather record:", weather.id if weather else None)
print("Soil record:", soil.id if soil else None)
print("DEM tile:", dem.id if dem else None)

# 3. Compute vegetation indices (NDVI & NDRE)
ndvi = compute_spectral_index("ndvi", nir=0.52, red=0.11)
ndre = compute_spectral_index("ndre", nir=0.52, rededge=0.24)
print("NDVI:", ndvi, "| NDRE:", ndre)